# Prep 4 · Probabilistic programming with NumPyro

**Time:** about 2 hours. **Needs:** Prep 1 and 3.

[NumPyro](https://num.pyro.ai/) lets you *write the generative story* — sample the unknowns, transform
them, observe the data — and then asks a sampler to invert it. Three primitives are all we use:

| primitive | meaning |
|---|---|
| `numpyro.sample("name", dist)` | an unknown with a prior |
| ordinary JAX code | deterministic transforms (the physics) |
| `numpyro.sample("name", dist, obs=data)` | the likelihood: data observed from `dist` |

Inference then comes in two flavours: **MAP** (one best answer, via optimisation) and **NUTS** (samples
from the whole posterior, via gradient-based MCMC). We redo the two problems from Prep 3 to check NumPyro
against the grid, then build a **one-dimensional MRI** — the exact structure of the school's
`recon_model`.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpyro
import numpyro.diagnostics
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, SVI, Trace_ELBO, autoguide

key = jax.random.PRNGKey(0)

## 1. The coin, in NumPyro

Compare with Prep 3: the prior is `Uniform(0, 1)`, the likelihood is `Binomial(10, p)` with the
observation `7` plugged in via `obs=`. That is the entire model. NUTS then draws samples of `p` from the
posterior; their mean should be the `2/3` you computed on the grid.

In [ ]:
def coin_model(n_heads, n_flips):
    p = numpyro.sample("p", dist.Uniform(0.0, 1.0))                       # prior
    numpyro.sample("heads", dist.Binomial(n_flips, probs=p), obs=n_heads)  # likelihood

mcmc = MCMC(NUTS(coin_model), num_warmup=500, num_samples=2000, progress_bar=False)
mcmc.run(key, n_heads=7, n_flips=10)
mcmc.print_summary()
p_samples = mcmc.get_samples()["p"]

pg = np.linspace(0, 1, 1001); grid_post = pg ** 7 * (1 - pg) ** 3; grid_post /= grid_post.sum() * (pg[1] - pg[0])
plt.hist(np.asarray(p_samples), bins=40, density=True, alpha=.5, label="NUTS samples"); plt.plot(pg, grid_post, label="grid posterior (Prep 3)")
plt.xlabel("p"); plt.legend(); plt.show()

assert abs(float(p_samples.mean()) - 2 / 3) < 0.03
print("NUTS agrees with the grid: mean =", round(float(p_samples.mean()), 3))

**Reading `print_summary`:** `mean`/`std` are the posterior moments; `r_hat` should be ≈ 1.00 and
`n_eff` comfortably in the hundreds — those two numbers are how you know the sampler can be trusted.
Keep an eye on them all week.

## 2. The two-pixel problem, in NumPyro

### Exercise 1 — write the model

Prior A from Prep 3: `x = (x₁, x₂)`, each `N(0, 1)`; the measurement `y = x₁ + x₂ + N(0, 0.1²)`, observed
as `1.0`. Write it with the primitives from the table above.

In [ ]:
def two_pixel_model(y_obs, sigma=0.1):
    x = numpyro.sample("x", dist.Normal(jnp.zeros(2), 1.0))      # prior A: independent N(0, 1)
    numpyro.sample("y", dist.Normal(x.sum(), sigma), obs=y_obs)  # the scanner measures the sum

<details><summary><b>Hint</b> — try the exercise first, then click to reveal</summary>

Two `numpyro.sample` calls. The latent: a site named `"x"` with a length-2 standard-normal prior —
`dist.Normal(jnp.zeros(2), 1.0)` gives two independent `N(0, 1)`s at once. The observation: a site
whose distribution is `dist.Normal(x.sum(), sigma)`, with the measured value plugged in via
`obs=y_obs`.

</details>

In [ ]:
mcmc = MCMC(NUTS(two_pixel_model), num_warmup=500, num_samples=2000, progress_bar=False)
mcmc.run(key, y_obs=1.0)
assert "x" in mcmc.get_samples(), "your model must have a sample site named 'x'"
xs = np.asarray(mcmc.get_samples()["x"])
mean, std = xs.mean(0), xs.std(0)

plt.figure(figsize=(4, 4)); plt.scatter(xs[:, 0], xs[:, 1], s=3, alpha=.3); plt.xlim(-2, 2); plt.ylim(-2, 2)
plt.xlabel("x1"); plt.ylabel("x2"); plt.title("posterior samples: the ridge, softened by the prior"); plt.show()
print("posterior mean", np.round(mean, 2), " std", np.round(std, 2), "  (grid in Prep 3 gave std ~ 0.70 each)")

# check
assert np.allclose(mean, [0.5, 0.5], atol=0.06)
assert np.all(np.abs(std - 0.707) < 0.08), "each pixel keeps ~0.71 of posterior std under prior A"
print("exercise 1 OK")

### MAP with `SVI` + `AutoDelta`

For the point estimate we don't sample — we optimise. `AutoDelta` is a "guide" that represents the
posterior by a single point; `SVI` moves that point to maximise the posterior. This is exactly how the
repo's `reconstruct_map` works.

In [ ]:
guide = autoguide.AutoDelta(two_pixel_model)
svi = SVI(two_pixel_model, guide, numpyro.optim.Adam(0.05), Trace_ELBO())
result = svi.run(key, 1000, y_obs=1.0, progress_bar=False)
x_map = result.params["x_auto_loc"]
print("MAP:", np.round(np.asarray(x_map), 3))
assert np.allclose(np.asarray(x_map), [0.5, 0.5], atol=0.02)

## 3. One-dimensional MRI

Now the real structure. A 1-D "image" `x` of 64 samples; the "scanner" measures its Fourier transform
`k = fft1c(x)` but only at the frequencies in a **mask** (a central band plus every 4th frequency), with
noise. We want `x` back.

The prior: `x = A z` with `z ~ N(0, I)`, where `A` is a fixed matrix whose columns are 12 smooth bumps.
So the prior says "x is a smooth combination of bumps" — a **hand-made decoder**. At the school the only
change is that `A z` becomes a *trained* neural network `decode(z)`.

In [ ]:
n, m = 64, 12
pos = np.arange(n)
centres = np.linspace(4, n - 4, m)
A = jnp.asarray(np.stack([np.exp(-0.5 * ((pos - c) / 3.0) ** 2) for c in centres], axis=1), dtype=jnp.float32)  # (64, 12)

def fft1c(x):   # centred, orthonormal 1-D FFT: the 1-D twin of the repo's fft2c
    return jnp.fft.fftshift(jnp.fft.fft(jnp.fft.ifftshift(x), norm="ortho"))
def ifft1c(k):
    return jnp.fft.fftshift(jnp.fft.ifft(jnp.fft.ifftshift(k), norm="ortho"))

k_true, k_noise, k_z = jax.random.split(jax.random.PRNGKey(3), 3)
z_true = jax.random.normal(k_z, (m,))
x_true = A @ z_true                                   # a signal the prior can represent

mask = np.zeros(n, np.float32); mask[::4] = 1; mask[n//2 - 4:n//2 + 4] = 1   # every 4th frequency + centre band
mask = jnp.asarray(mask)
sigma = 0.05
noise = sigma * (jax.random.normal(k_noise, (n,)) + 1j * jax.random.normal(k_true, (n,)))
y = mask * fft1c(x_true) + mask * noise               # what we measure

x_zf = ifft1c(y).real                                 # zero-filled: the baseline
print(f"measured {int(mask.sum())} of {n} frequencies (R_eff = {n / float(mask.sum()):.1f})")
plt.plot(x_true, "k", label="truth"); plt.plot(x_zf, label="zero-filled"); plt.legend(); plt.show()

### Exercise 2 — the reconstruction model

Write `recon_1d`, the generative story of the measurement: a standard-normal prior on the latent `z`,
the hand-made decoder turning `z` into a signal, the masked Fourier transform, and finally observing
the **real and imaginary parts** of `y_obs` at the measured frequencies (a complex Gaussian likelihood
is just two real ones).

One landmine you could not guess: switch the likelihood off at unmeasured frequencies with
`.mask(obs)` on the distribution (where `obs = mask.astype(bool)`), **not** with boolean indexing like
`k.real[obs]` — under NUTS the mask is a traced array and indexing fails.

In [ ]:
def recon_1d(y_obs, mask, A, sigma):
    z = numpyro.sample("z", dist.Normal(jnp.zeros(A.shape[1]), 1.0))   # prior on the latent
    x = A @ z                                                          # the (hand-made) decoder
    k = mask * fft1c(x)                                                # forward model
    obs = mask.astype(bool)
    numpyro.sample("y_re", dist.Normal(k.real, sigma).mask(obs), obs=y_obs.real)   # likelihood on the
    numpyro.sample("y_im", dist.Normal(k.imag, sigma).mask(obs), obs=y_obs.imag)   # measured entries only

<details><summary><b>Hint</b> — try the exercise first, then click to reveal</summary>

The three deterministic lines mirror section 3's cell: `z` is a length-`A.shape[1]` standard normal
(same pattern as exercise 1's latent), `x = A @ z`, and `k = mask * fft1c(x)`.

</details>

<details><summary><b>Bigger hint</b> — try the exercise first, then click to reveal</summary>

Each observed site pairs one real part of the data with the matching part of the model's k-space:
`numpyro.sample("y_re", dist.Normal(k.real, sigma).mask(obs), obs=y_obs.real)` — and the same again
with `.imag` for `"y_im"`.

</details>

In [ ]:
mcmc = MCMC(NUTS(recon_1d), num_warmup=500, num_samples=1000, progress_bar=False)
mcmc.run(key, y, mask, A, sigma)
assert "z" in mcmc.get_samples(), "your model must have a sample site named 'z'"
zs = mcmc.get_samples()["z"]
x_samples = jax.vmap(lambda z: A @ z)(zs)             # decode every posterior sample
x_mean, x_std = x_samples.mean(0), x_samples.std(0)

err = lambda est: float(jnp.linalg.norm(est - x_true) / jnp.linalg.norm(x_true))
plt.fill_between(pos, x_mean - 2 * x_std, x_mean + 2 * x_std, alpha=.25, label="posterior ± 2 std")
plt.plot(x_true, "k", label="truth"); plt.plot(x_zf, alpha=.6, label=f"zero-filled (err {err(x_zf):.2f})"); plt.plot(x_mean, label=f"posterior mean (err {err(x_mean):.2f})")
plt.legend(); plt.show()

# check
r_hat = np.asarray(numpyro.diagnostics.summary(mcmc.get_samples(), group_by_chain=False)["z"]["r_hat"])
assert err(x_mean) < err(x_zf), "the posterior mean must beat zero-filling"
assert err(x_mean) < 0.2
assert np.all(r_hat < 1.05), "r_hat should be ~1: the chain has converged"
print("exercise 2 OK")

### MAP for the same model

Same trick as before: `AutoDelta` + `SVI`. For this model everything is Gaussian, so the MAP and the
posterior mean coincide — but the MAP gives you no uncertainty band. That band is the reason the school project goes all the way to NUTS.

In [ ]:
guide = autoguide.AutoDelta(recon_1d)
svi = SVI(recon_1d, guide, numpyro.optim.Adam(0.05), Trace_ELBO())
result = svi.run(key, 4000, y, mask, A, sigma, progress_bar=False)
x_map = A @ result.params["z_auto_loc"]
print(f"error: zero-filled {err(x_zf):.3f} · MAP {err(x_map):.3f} · posterior mean {err(x_mean):.3f}")
assert err(x_map) < err(x_zf)

## What changes at the school

```python
def recon_model(y_obs, mask, decode, latent_dim, sigma):    # src/mrigen/recon/vae_numpyro.py
    z = numpyro.sample("z", dist.Normal(jnp.zeros(latent_dim), 1.0))
    x = decode(z)             # <- A @ z  becomes a trained decoder
    k = mask * fft2c(x)       # <- fft1c becomes fft2c, 64 samples become 128 x 128
    ...                       # the likelihood is identical
```

You have already written this. The next notebook replaces `A` with a decoder that *learned* what the
signals look like — and that is the whole project, in one dimension.

## Done when

- all checks print OK;
- you can explain the three primitives, what `obs=` does, and why `.mask(obs)` rather than indexing;
- you know which two numbers in `print_summary` tell you whether to trust the sampler.